## DATA 442 / 610 - Deep Learning and Neural Networks
## Assignment 1: Building a NN from Scratch
## Mark J Serena - W&M ID # 930657215

In [ ]:
from collections import Counter
from pathlib import Path

import datetime
import math
import os
import random
import sys
from operator import itemgetter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn

from sklearn.preprocessing import PolynomialFeatures
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix,classification_report
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.metrics import roc_curve, auc

print(f"Python {sys.version}")
print(f"sklearn {sklearn.__version__}")

In [ ]:
import numpy as np
def make_dataset(N=2048, random_seed=42):
    np.random.seed(random_seed )
    y = np.random.choice(2, size=N).astype(np.uint8)
    X = np.empty([N, 4], dtype=np.float32)
    X[:, 0] = np.random.uniform(low=-5, high=5, size=N)
    X[:, 1] = np.random.normal(loc=0., scale=2.5 , size=N)
    X[:, 2] = np.random.normal(loc=0., scale=1., size=N)
    X[:, 3] = np.sqrt(X[:, 0]**2 + X[:, 1]**2 + X[:, 2]**2 + y **2)
    X[:, 3] *= np.random.choice([-1, 1], size=N)
    return X, y

## Problem 2 (50 points) - Hyperparameter tuning exploration

### Part 1: Data Preprocessing (4 points)

### Question 1 (2 points): 

Visualize the dataset by plotting pairwise correlations between each input feature for each class.


In [ ]:
# Use provided function to create the variables
X, y = make_dataset()

# Column names for the four input features.
features = ["x0", "x1", "x2", "x3"]
# Create dataframe
df = pd.DataFrame(X, columns=features)
# Add a column for the dependent variable 
df["class"] = y

print(df.describe())
print()
sns.pairplot(
    df, vars=features, hue="class", plot_kws={"alpha": 0.5}
)

plt.legend
plt.savefig('Assignment1_P2_Part1_Q1.png')
plt.show()

### Question 2 (2 points): Describe the structure of the dataset. Do you think it is linearly separable?

Linearly separable? No. Too much overlap.

The fourth feature in the dataset, called x3 here, does show some interesting behavior. It does seem to divide into two distinct clusters although the target variable is clearly mixed in both of the two clusters. Looking at the function that created x3 you can see that it was randomly multiplied by 1 or -1. This caused the divsion into two clusters. 

The plots for x3 seemed to involve curves. Curves alone do not necessarily imply a quadratic relationship, but.... Looking at the function that generated the data you can see that x3 is generated using a square root of a sum of squares. Squaring x3 reveals the quadratic relationship. So logistic regression with degree-2 polynomial features could separate.

### Part 2: Polynomial Logistic Regression (6 points)


### Question 1 (2 points): 

Implement a single-output logistic regression model fit with mini-batch stochastic gradient descent to minimize the cross-entropy loss. Include an argument degree that allows polynomial feature augmentation. You may use the notebook from class as a template.

In [ ]:
class LogisticRegression:
    """
    Binary logistic regression trained with full-batch gradient descent.

    Parameters
    ----------
    alpha : float
        Learning rate (step size for each gradient update).
    num_epochs : int
        Number of full passes over the training data.
    random_seed : int
        Seed for weight initialization, so results are reproducible.

    Attributes
    ----------
    w_ : ndarray, shape [n_features,]
        Learned feature weights (set by `fit`).
    b_ : float
        Learned bias term (set by `fit`).
    loss : list of float
        Cross-entropy loss recorded at each epoch.
    """

    def __init__(self,
                 alpha=0.25,
                 num_epochs=500,
                 random_seed=42):
        # Store hyperparameters only; learned parameters are created in fit().
        self.alpha = alpha
        self.num_epochs = num_epochs
        self.random_seed = random_seed

    def fit(self, X, y):
        """Train the model on the given data.

        X : ndarray with shape [n_examples, n_features]
        y : ndarray with shape [n_examples,], binary labels (0 or 1)

        Returns self, so calls can be chained (e.g., model.fit(X, y).score(X, y)).
        """
        # Seed the RNG so the random initialization is reproducible.
        np.random.seed(self.random_seed)

        # Initialize weights to small random values near zero and the bias
        # to zero. Small weights keep the initial predictions close to 0.5.
        self.w_ = np.random.normal(loc=0, scale=0.01, size=X.shape[1])
        self.b_ = np.float32(0.)

        # Track the loss at each epoch to monitor convergence.
        self.loss = []

        # Full-batch gradient descent: each epoch uses the entire dataset.
        for _ in range(self.num_epochs):
            loss = self._fit_epoch(X, y)
            self.loss.append(loss)

        return self

    def _fit_epoch(self, X, y):
        """Run one gradient descent step over the full dataset and
        return the cross-entropy loss for this epoch."""
        # Forward pass: linear combination, then sigmoid to get probabilities.
        z_pred = self._pre_activation(X)
        y_pred = self._activation(z_pred)

        # Prediction error. For logistic regression with cross-entropy loss,
        # the gradient of the loss with respect to z simplifies to
        # (y_pred - y), so `diff` is the negative gradient.
        diff = y - y_pred

        # Because `diff` is the negative gradient, adding it is equivalent
        # to descending the loss. Dividing by n_examples averages the
        # gradient over the batch.
        self.w_ += self.alpha * diff.dot(X) / X.shape[0]
        self.b_ += self.alpha * diff.sum() / X.shape[0]

        # Loss is computed from the predictions made before this update.
        loss = self._cross_entropy(y, y_pred)
        return loss

    def _cross_entropy(self, y_true, y_pred, eps=1e-10):
        """Binary cross-entropy averaged over all examples.

        `eps` is added inside the logs to avoid log(0) = -inf when a
        prediction is exactly 0 or 1.
        """
        return -np.mean(
            y_true * np.log(y_pred + eps) + \
            (1 - y_true) * np.log(1 - y_pred + eps)
        )

    def _pre_activation(self, X):
        """Compute the linear part of the model: z = Xw + b."""
        return np.dot(X, self.w_) + self.b_

    def _activation(self, z):
        """Sigmoid function, mapping z to a probability in (0, 1).

        Clipping z to [-250, 250] prevents overflow in np.exp for
        very large negative inputs.
        """
        return 1. / (1 + np.exp(-np.clip(z, -250, 250)))

    def predict(self, X):
        """Return predicted class labels (0 or 1) for each example in X."""
        # threshold should be the midpoint between the two classes values
        # in this case 0.5 is between 0 and 1
        threshold = 0.5
        z_pred = self._pre_activation(X)
        y_pred = self._activation(z_pred)
        return np.where(y_pred >= threshold, 1, 0)

    def score(self, X, y):
        """Return classification accuracy: the fraction of correct predictions."""
        return np.mean(self.predict(X) == y)

In [ ]:
def shuffle(X, y):
    """Shuffle training data each epoch.

    Applies the same random permutation to X and y so each example stays
    paired with its label. Returns shuffled copies; the inputs are unchanged.
    """
    idx = np.random.permutation(X.shape[0])
    return X[idx, :], y[idx]


class LogisticRegressionPolynomial(LogisticRegression):
    """
    Logistic regression on polynomial features, trained with mini-batch
    gradient descent.

    Extends LogisticRegression by expanding the inputs into polynomial terms
    (e.g., x1, x2, x1^2, x1*x2, x2^2 for degree 2), which lets the model
    learn a nonlinear decision boundary.

    Parameters
    ----------
    degree : int
        Maximum degree of the polynomial features.
    batch_size : int
        Number of examples per gradient update.
    **kwargs
        Passed to LogisticRegression (alpha, num_epochs, random_seed).

    Attributes
    ----------
    poly_ : PolynomialFeatures
        Fitted feature transformer (set by `fit`).
    w_, b_ : learned weights (one per polynomial feature) and bias.
    loss : list of float
        Training cross-entropy recorded after each epoch.
    accuracy : list of float
        Training accuracy recorded after each epoch.
    """

    def __init__(self, degree=1, batch_size=16, **kwargs):
        # Store this subclass's hyperparameters, then let the parent class
        # handle alpha, num_epochs, and random_seed.
        self.degree = degree
        self.batch_size = batch_size
        super().__init__(**kwargs)

    def fit(self, X, y, X_val, y_val):
        """Train the model on the given data.

        X : ndarray with shape [n_examples, n_features] (raw, unexpanded)
        y : ndarray with shape [n_examples,], binary labels (0 or 1)
        X_val : same shape as training
        y_val : same shape as training

        Returns self.
        """
        # Seed the RNG so both weight initialization and the per-epoch
        # shuffling are reproducible.
        np.random.seed(self.random_seed)

        # Build the polynomial transformer. include_bias=False because the
        # model already has its own bias term, b_. Fitting only determines
        # the number of input features and the terms to generate.
        self.poly_ = PolynomialFeatures(degree=self.degree, include_bias=False)
        self.poly_.fit(X)

        print(f'Degree {self.degree} augmented feature set contains {self.poly_.n_output_features_} features')

        # One weight per *expanded* feature, not per original feature.
        self.w_ = np.random.normal(loc=0, scale=0.01,
                                   size=self.poly_.n_output_features_)
        self.b_ = np.float32(0.)

        # Track loss and accuracy each epoch to monitor training progress.
        self.loss = []
        self.accuracy = []
        # Track loss and accuracy each epoch to monitor validation
        self.val_loss = []
        self.val_accuracy = []

        # iterate over epochs
        for _ in range(self.num_epochs):
            loss = self._fit_epoch(X, y)
            self.loss.append(loss)
            # score() calls predict(), which expands X itself, so raw X is
            # passed here.
            self.accuracy.append(self.score(X, y))

            # Compute validation accuracy
            # Store validation accuracy
            self.val_accuracy.append(self.score(X_val, y_val))
            # For validation loss, obtain the predicted probabilities, 
            # compare them with y_val using cross-entropy, 
            # and store the result
            # Compute validation probabilities without updating weights.
            X_val_prep = self._preprocess(X_val)
            z_val = self._pre_activation(X_val_prep)
            y_val_prob = self._activation(z_val)
            
            # Record this epoch's validation loss.
            val_loss = self._cross_entropy(y_val, y_val_prob)
            self.val_loss.append(val_loss)

        return self

    def _preprocess(self, X):
        """Expand raw features into polynomial features."""
        return self.poly_.transform(X)

    def _fit_epoch(self, X, y):
        """Run one epoch of mini-batch gradient descent and return the
        training loss measured after the epoch's final update."""
        # Reshuffle each epoch so batches differ from epoch to epoch.
        X_shuf, y_shuf = shuffle(X, y)

        # Step through the data in chunks of batch_size. The last batch may
        # be smaller if n_examples isn't divisible by batch_size.
        for batch_start in range(0, X.shape[0], self.batch_size):
            batch_end = batch_start + self.batch_size
            xb = X_shuf[batch_start:batch_end, :]
            yb = y_shuf[batch_start:batch_end]

            # Forward pass on the batch: expand features, linear combination,
            # then sigmoid.
            X_prep = self._preprocess(xb)
            z_pred = self._pre_activation(X_prep)
            y_pred = self._activation(z_pred)

            # Negative gradient of the loss w.r.t. z (see parent class).
            diff = yb - y_pred

            # Update parameters using the batch-averaged gradient. Using
            # xb.shape[0] (not batch_size) keeps the average correct for the
            # smaller final batch.
            self.w_ += self.alpha * diff.dot(X_prep) / xb.shape[0]
            self.b_ += self.alpha * diff.mean()

        # Evaluate training loss after all batches have updated the model.
        X_prep = self._preprocess(X)
        z_pred = self._pre_activation(X_prep)
        y_pred = self._activation(z_pred)
        loss = self._cross_entropy(y, y_pred)
        return loss

    def predict(self, X):
        """Return predicted class labels (0 or 1) for raw (unexpanded) X."""
        # Expand the features first, then reuse the parent's thresholding.
        X_prep = self._preprocess(X)
        return super().predict(X_prep)

### Question 2 (2 points): 

Fit the model with degree=1 and degree=2. Be sure to track loss and accuracy for both training and validation sets, and retain a held-out test set of at least 20% of the overall dataset.

In [ ]:
from sklearn.model_selection import train_test_split

# Split into 80/20
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Split the 80 into 75/25 (60% of original / 20% of original)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val,
    test_size=0.25, random_state=42, stratify=y_train_val
)

In [ ]:
# MLP requires the input variables have a shape of 
# [number of observations, number of features]
# Reshape to convert [number of observations, ]
# to [number of observations, 1]
y_train_nn = y_train.reshape(-1, 1)
y_val_nn = y_val.reshape(-1, 1)
y_test_nn = y_test.reshape(-1, 1)

In [ ]:
# Create model of 1 degree
lrp_model1 = LogisticRegressionPolynomial(degree=1)
# Fit that model using training data, 
# evaluate it on X_val, y_val after each epoch, recording 
# validation loss and accuracy without updating weights 
# from validation data.
lrp_model1.fit(X_train, y_train, X_val, y_val)

# Create model of 2 degree
lrp_model2 = LogisticRegressionPolynomial(degree=2)
# Fit that model using training data, 
# evaluate it on X_val, y_val after each epoch, recording 
# validation loss and accuracy without updating weights 
# from validation data.
lrp_model2.fit(X_train, y_train, X_val, y_val)

# Obtain training score
train_score = lrp_model1.score(X_train, y_train)
print(f"LogisticRegressionPolynomial(degree=1) training score {train_score}")
# Obtain training score
train_score = lrp_model2.score(X_train, y_train)
print(f"LogisticRegressionPolynomial(degree=2) training score {train_score}")

print()

# Obtain validation score
val_score = lrp_model1.score(X_val, y_val)
print(f"LogisticRegressionPolynomial(degree=1) validation score {val_score}")
# Obtain validation score
val_score = lrp_model2.score(X_val, y_val)
print(f"LogisticRegressionPolynomial(degree=2) validation score {val_score}")

### Question 3 (2 points): 

For each model, plot the loss and accuracy curves for both training and
validation sets. Also plot a confusion matrix and ROC-AUC curve on the held-out test set. Which
model performs better? Why do you think this is the case?

In [ ]:
ax = plt.figure().gca()
ax.plot(lrp_model1.loss, label='LogisticRegressionPolynomial(degree=1)')
ax.plot(lrp_model2.loss, label='LogisticRegressionPolynomial(degree=2)')
ax.legend()
ax.set(xlabel='Epochs', ylabel='Cross-entropy Loss', yscale='linear')

plt.title("Tracking Loss - Training")
plt.savefig("Assignment1_P2_Part2_Q2a.png")
plt.show()

In [ ]:
ax = plt.figure().gca()
ax.plot(lrp_model1.accuracy, label='LogisticRegressionPolynomial(degree=1)')
ax.plot(lrp_model2.accuracy, label='LogisticRegressionPolynomial(degree=2)')
ax.legend()
ax.set(xlabel='Epochs', ylabel='Accuracy', yscale='linear')

plt.title("Tracking Accuracy - Training")
plt.savefig("Assignment1_P2_Part2_Q2b.png")
plt.show()

In [ ]:
ax = plt.figure().gca()
ax.plot(lrp_model1.val_loss, label='LogisticRegressionPolynomial(degree=1)')
ax.plot(lrp_model2.val_loss, label='LogisticRegressionPolynomial(degree=2)')
ax.legend()
ax.set(xlabel='Epochs', ylabel='Cross-entropy loss', yscale='linear')

plt.title("Validation")
plt.savefig("Assignment1_P2_Part2_Q2c.png")
plt.show()

In [ ]:
ax = plt.figure().gca()
ax.plot(lrp_model1.val_accuracy, label='LogisticRegressionPolynomial(degree=1)')
ax.plot(lrp_model2.val_accuracy, label='LogisticRegressionPolynomial(degree=2)')
ax.legend()
ax.set(xlabel='Epochs', ylabel='Accuracy', yscale='linear')

plt.title("Tracking Accuracy - Validation")
plt.savefig("Assignment1_P2_Part2_Q2d.png")
plt.show()

In [ ]:
# List of models to be compared
# Tuple with the degree and model name
models = [
    (1, lrp_model1),
    (2, lrp_model2),
]

# Create a figure with 2 rows and 2 columns
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Iterate over list of models
# row gets its value from enumerate(models), 
# which pairs each item in models with a counter starting at 0.
for row, (degree, model) in enumerate(models):

    # Create probabilities
    X_test_prep = model._preprocess(X_test)
    y_test_prob = model._activation(model._pre_activation(X_test_prep))

    
    # Create predictions from test data
    y_test_pred = model.predict(X_test)

    # Create a confusion matrix
    # comparing predictions to actuals
    cm = confusion_matrix(y_test, y_test_pred, labels=[0, 1])
    # Display confusion matrix
    ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=[0, 1]
    ).plot(ax=axes[row, 0], colorbar=False)
    # Set title of confusion matrix (always in column 0)
    axes[row, 0].set_title(f"Degree {degree}: Test Confusion Matrix")

    # Create ROC using probabilities
    fpr, tpr, thresholds = roc_curve(y_test, y_test_prob)
    roc_auc = auc(fpr, tpr)
    # Place ROC in coliumn 1
    axes[row, 1].plot(
        fpr, tpr, label=f"AUC = {roc_auc:.3f}"
    )
    axes[row, 1].plot([0, 1], [0, 1], "k--")
    axes[row, 1].set(
        xlabel="False Positive Rate",
        ylabel="True Positive Rate",
        title=f"Degree {degree}: Test ROC Curve"
    )
    axes[row, 1].legend(loc="lower right")

plt.tight_layout()
plt.savefig("Assignment1_P2_Part2_Q3.png")
plt.show()

### Question 3 (2 points):  Also plot a confusion matrix and ROC-AUC curve on the held-out test set. 

Which model performs better? Why do you think this is the case?

From the plots shown above, clearly degree = 2 is better, actually perfect on this held-out test set. The reason seems to be that at degree 2, the model has the features it needs to accurately make predictions. 

We saw earlier that the plots involving x3 seemed to show curves. Curves alone do not necessarily guarantee a quadratic relationship, but they do tend to hint at one. Here we can also look at the function that generated the data.

In that function, the feature x3 is generated using a square root of a sum of squares. Squaring reveals the quadratic relationship. Degree-2 polynomial features include these squared terms, allowing logistic regression to capture that relationship and separate the classes. Using just degree = 1 cannot capture that curved boundary.

In [ ]:
degrees = [1,2,3,4,5,6,7,8,9]
test_accuracies = []
test_aucs = []

for i in degrees:
    # Create model of i degree
    lrp_modelx = LogisticRegressionPolynomial(degree=i)
    # Fit that model using training data, 
    # evaluate it on X_val, y_val after each epoch, recording 
    # validation loss and accuracy without updating weights 
    # from validation data.
    lrp_modelx.fit(X_train, y_train, X_val, y_val)
    
    # Obtain test score
    test_score = lrp_modelx.score(X_test, y_test)
    test_accuracies.append(test_score)


    # Expand test features to this model's polynomial degree.
    X_test_prep = lrp_modelx._preprocess(X_test)
    
    # Apply the learned weights and bias.
    z_test = lrp_modelx._pre_activation(X_test_prep)
    
    # Convert to probabilities using sigmoid.
    y_test_prob = lrp_modelx._activation(z_test)
    
    # Create ROC and calculate AUC.
    fpr, tpr, thresholds = roc_curve(y_test, y_test_prob)
    roc_auc = auc(fpr, tpr)
    # Create ROC using probabilities
    fpr, tpr, thresholds = roc_curve(y_test, y_test_prob)
    roc_auc = auc(fpr, tpr)
    test_aucs.append(roc_auc)



In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(
    degrees, test_accuracies,
    label="Test Accuracy",
    marker='o', linestyle='-')

# add in the values
for i in range(len(test_accuracies)):
    plt.text(i+1, test_accuracies[i]+0.01, str(round(test_accuracies[i],3)), 
             horizontalalignment = 'center',             
             fontsize = 10)
    
plt.xlabel("Number of Degrees")
plt.ylabel("Test Accuracy")
plt.grid(True)
plt.title(f"Test Accuracy by Number of Degrees")
plt.show()

plt.figure(figsize=(10, 5))

plt.plot(
    degrees, test_aucs,
    label="Test ROC_AUC",
    marker='o', linestyle='-')

# add in the values
for i in range(len(test_aucs)):
    plt.text(i+1, test_aucs[i]+0.01, str(round(test_aucs[i],3)), 
             horizontalalignment = 'center',             
             fontsize = 10)
    
plt.xlabel("Number of Degrees")
plt.ylabel("Test ROC-AUC")
plt.grid(True)
plt.title(f"Test ROC-AUC by Number of Degrees")
plt.show()

### Are there any patterns? If so, why do you think they occur?

Yes, there's definitely a pattern. The models with even degrees, 2,4,6,... perform better but increasing degree does not consistently improve accuracy.  In-spite of increasing the the number of features included.

Every model of degree 2 or greater includes the squared terms needed to separate the classes but when the degree is odd its not able to accurately make a prediction. Does the model have difficulty learning the correct weights to apply? No scaling or regularization was used. The degree specifies the maximum total power of the polynomial features. It doesn't mean that power is applied to all features, but the unscaled inputs at higher degrees can produce very large feature values. With the same learning rate, these can cause large weight updates and make training less stable.

But,.... that doesn't explain the even-odd variation. In data science its either the model or the data. In this case I'm not sure which.

In [ ]:
# Create ROC using probabilities
fpr, tpr, thresholds = roc_curve(y_test, y_test_prob)
roc_auc = auc(fpr, tpr)
plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.3f}")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title(f"Degree {degree}: Test ROC Curve")
plt.legend(loc="lower right")

## Part 3: Single-layer NN (14 points)


### Question 1 (5 points): 

Implement a neural network with one hidden layer. For this neural network, use the ReLU activation on the hidden layers and the tanh activation on the output layer

$$
tanh(z) = \frac{e^z - e^{-z}}{e^z + e^{-z}}
$$

**Derive d_h_output__d_z_output and implement the backward pass for this neural network.** 

The output activation is:

    h_output = tanh(z_output)

Rules for derivative of e to a power: 
$$\frac{d}{dz}e^{z} = e^{z}$$

$$\frac{d}{dz}e^{-z} = -e^{-z}$$

And the quotient rule is:

$$
f'(z) = \frac{u'(z)v(z) - u(z) v'(z)}{v(z)^{2}}
$$

Let the numerator be $u(z)$ and the denominator be $v(z)$

$$u(z) = e^{z} - e^{-z}$$

$$v(z) = e^{z} + e^{-z}$$

And their derivatives are: 
$$
u'(z) = e^{z} + e^{-z}
$$
$$
v'(z) = e^{z} - e^{-z}
$$


Now substitute:
$$
tanh'(z) = \frac{(e^{z} + e^{-z})(e^{z} + e^{-z}) - (e^{z} - e^{-z})(e^{z} - e^{-z})} {(e^{z} + e^{-z})^{2}}
$$

Keep going:
$$
tanh'(z) = \frac{(e^{z} + e^{-z})^{2} - (e^{z} - e^{-z})^{2}} {(e^{z} + e^{-z})^{2}}
$$

More:
$$
tanh'(z) = 1 - (\frac{e^z - e^{-z}}{e^z + e^{-z}})^{2}
$$


Therefore:
$$
tanh'(z) = 1 - tanh^{2}(z)
$$

Since
$$
h_{output} = tanh(z_{output})
$$

$$
\frac{\partial h_{output}}{\partial z_{output}} = 1 - h^{2}_{output}
$$

In my implementation, h_output is named a_output, so:

    d_a_output__d_z_output = 1 - a_output**2



In [ ]:
# Functions used by MLP_2Layer_Relu and its train() function

def preactivation(x, w, b):
    """ Applies weights and biases
    """
    return np.dot(x, w) + b

def grad_preactivation(x, w, b, z):
    """ Accepts inputs x, w, b, output z
        Returns gradient of output w.r.t. each input
    """
    dz_dx = w
    dz_dw = x
    dz_db = 1.
    return dz_dx, dz_dw, dz_db

def relu(z):
    """ Thresholded ramp activation function
    """
    result = np.maximum(0, z)
    return result

# Derivative of relu
def grad_relu(z, a):
    """ Accepts input z, output a
        Returns gradient of output w.r.t. input
    """
    result = np.where(z >= 0, 1., 0.)
    return result

def mse_loss(y_true, y_pred):
    """ Mean squared error 
    """
    return np.mean(
        np.power(y_true - y_pred, 2).sum(axis=1)
    )

def grad_mse_loss(y_true, y_pred):
    """ Returns gradient of MSE loss w.r.t. model prediction y_pred
    """
    return 2. / y_true.shape[0] * (y_pred - y_true)



In [ ]:
def accuracy(y_true, y_pred):
    threshold = 0.5
    # Extract the labels from the single column.
    true_labels = y_true[:, 0]

    # Convert each output to class 0 or 1.
    predicted_labels = (y_pred[:, 0] >= threshold).astype(int)
    print(predicted_labels)
    # Calculate the fraction of correct predictions.
    correct_predictions = true_labels == predicted_labels
    accuracy_score = np.mean(correct_predictions)

    return accuracy_score 

In [ ]:
class MLP_2Layer_ReLU:
    def __init__(self, 
                 num_inputs,
                 num_hidden,
                 num_outputs,
                 random_seed=42):
        self.num_inputs = num_inputs
        self.num_hidden = num_hidden
        self.num_outputs = num_outputs
        self.random_seed = random_seed
        self._initialize_weights()

    def _initialize_weights(self):
        np.random.seed(self.random_seed)

        # Initialize hidden layer weights and biases
        self.w_hidden = np.random.normal(loc=0, scale=0.1,
                                         size=(self.num_inputs, self.num_hidden))
        self.b_hidden = np.zeros(self.num_hidden)

        # Initialize output layer weights and biases
        self.w_output = np.random.normal(loc=0, scale=0.1,
                                         size=(self.num_hidden, self.num_outputs))
        self.b_output = np.zeros(self.num_outputs)

    def forward(self, x):
        """ Feed input data x through each network layer 
            Return a final network output, as well as the intermediate
            values at each layer
        """
        # Apply hidden layer functions
        z_hidden = preactivation(x, self.w_hidden, self.b_hidden)
        # Using relu
        a_hidden = relu(z_hidden)

        # Apply output layer function
        z_output = preactivation(a_hidden, self.w_output, self.b_output)
        # Using tanh for activation
        a_output = np.tanh(z_output)


        return a_output, (
            a_output,
            z_output,
            a_hidden,
            z_hidden,
        )

    def backward(self, x, y_true, a_output, z_output, a_hidden, z_hidden):
        """ Compute gradients with respect to each model parameter """
        # Sanity check on input shapes
        B, I = x.shape # [B, I]
        _, O = y_true.shape # [B, O]
        assert y_true.shape[0] == B

        # Compute gradients w.r.t. output layer preactivation
        # Should be same size as z_output: [B, O]
        d_loss__d_a_output = grad_mse_loss(y_true, a_output) #[B, O]

        # Derivative of tanh
        d_a_output__d_z_output = 1.0 - a_output**2  # Shape: (B, O)

        d_loss__d_z_output = d_loss__d_a_output * d_a_output__d_z_output #[B, O]

        # Compute gradients w.r.t. output layer weights
        # Should be same size as w_output: [H, O]
        d_z_output__d_w_output = a_hidden #[B, H]
        d_loss__d_w_output = np.einsum('Bo,Bh->ho', d_loss__d_z_output, d_z_output__d_w_output)

        # Compute gradients w.r.t. output layer biases
        # Should be same size as b_output: [O,]
        d_z_output__d_b_output = 1.
        d_loss__d_b_output = np.einsum('Bo,->o', d_loss__d_z_output, d_z_output__d_b_output)

        # Compute gradients w.r.t. hidden layer preactivation
        # Should be same size as z_hidden: [B, H]
        d_z_output__d_a_hidden = self.w_output #[H, O]
        d_loss__d_a_hidden = np.einsum('Bo,ho->Bh', d_loss__d_z_output, d_z_output__d_a_hidden)

        # Using grad_relu
        # Derivative of relu
        d_a_hidden__d_z_hidden = grad_relu(z_hidden, a_hidden) #[B, H]
        d_loss__d_z_hidden = d_loss__d_a_hidden * d_a_hidden__d_z_hidden #[B, H]

        # Compute gradients w.r.t. hidden layer weights
        # Should be same size as w_hidden: [I, H]
        d_z_hidden__d_w_hidden = x #[B, I]
        d_loss__d_w_hidden = np.einsum('Bh,Bi->ih', d_loss__d_z_hidden, d_z_hidden__d_w_hidden)

        # Compute gradients w.r.t. hidden layer biases
        # Should be same size as b_hidden: [H,]
        d_z_hidden__d_b_hidden = 1.
        d_loss__d_b_hidden = np.einsum('Bh,->h', d_loss__d_z_hidden, d_z_hidden__d_b_hidden)

        # We're done. Return all the gradients w.r.t. weights and biases
        return d_loss__d_w_output, \
               d_loss__d_b_output, \
               d_loss__d_w_hidden, \
               d_loss__d_b_hidden

    def weight_update(self, d_loss__d_w_output, d_loss__d_b_output, d_loss__d_w_hidden, d_loss__d_b_hidden, lr=0.3):
        """ Update weights and biases via gradient descent
        """
        self.w_output -= lr * d_loss__d_w_output
        self.b_output -= lr * d_loss__d_b_output
        self.w_hidden -= lr * d_loss__d_w_hidden
        self.b_hidden -= lr * d_loss__d_b_hidden

In [ ]:
def batch_loader(X, y, batch_size=32):
    """Shuffle observations and return one batch at a time."""

    indices = np.arange(X.shape[0])
    np.random.shuffle(indices)

    for start in range(0, len(indices), batch_size):
        batch_indices = indices[start:start + batch_size]
        yield X[batch_indices], y[batch_indices]

In [ ]:
def train(model,
          X_train,
          y_train,
          X_val,
          y_val,
          num_epochs=1000,
          lr=0.01,
          batch_size=32):
    """Train one model and record its performance after each epoch."""

    # Create lists to store results
    train_loss, train_acc = [], []
    val_loss, val_acc = [], []

    # Iterate thru the epochs
    for epoch in range(1, num_epochs + 1):

        # Shuffle the training data and process each batch.
        for X_batch, y_batch in batch_loader(X_train, y_train, batch_size):

            # Forward pass: calculate predictions and intermediate values.
            predictions, layer_values = model.forward(X_batch)

            # Backward pass: calculate weight and bias gradients.
            gradients = model.backward(X_batch, y_batch, *layer_values)

            # Update the weights and biases.
            model.weight_update(*gradients, lr=lr)

        # Evaluate the training set after this epoch.
        train_predictions, _ = model.forward(X_train)
        train_loss.append(mse_loss(y_train, train_predictions))
        train_acc.append(accuracy(y_train, train_predictions))

        # Evaluate the validation set after this epoch.
        val_predictions, _ = model.forward(X_val)
        val_loss.append(mse_loss(y_val, val_predictions))
        val_acc.append(accuracy(y_val, val_predictions))

        # Optional: print a brief update every 100 epochs.
        if epoch % 200 == 0 or epoch == num_epochs:
            print(
                f"\tEpoch {epoch}/{num_epochs} | "
                f"\tTraining accuracy: {train_acc[-1]:.3f} | "
                f"\tValidation accuracy: {val_acc[-1]:.3f}"
            )

    return train_loss, train_acc, val_loss, val_acc

In [ ]:
def accuracy(y_true, y_pred):
    # Threshold between 0 and 1
    threshold = 0.5
    # Extract the labels from the single column.
    true_labels = y_true[:, 0]

    # Convert each output to class 0 or 1.
    predicted_labels = (y_pred[:, 0] >= threshold).astype(int)

    # Calculate the fraction of correct predictions.
    correct_predictions = true_labels == predicted_labels
    accuracy_score = np.mean(correct_predictions)

    return accuracy_score

### Question 2 (5 points): Train neural network models with hidden sizes of  [16, 32, 64, 128, 256, 512, 1024]. 

Be sure to record training and validation loss/accuracy curves, and use the same test
dataset from Part 2. 

Use following hyperparameters:
- batch size = 32
- epochs = 1000
- learning rate = 0.01

In [ ]:
# Create dataframe to store results
df = pd.DataFrame(columns=['activation', 'hidden_size', 
                           'train_mse', 
                           'train_acc', 
                           'val_mse', 
                           'val_acc',
                           'test_mse',
                           'test_acc',
                           'y_test_pred'])

In [ ]:
# Iterate thru hidden size values
for hidden_size in [16, 32, 64, 128, 256, 512, 1024]:
    activation = "ReLU"
    print(f"{activation} MLP w/Hidden Size {hidden_size}")
    model = MLP_2Layer_ReLU(num_inputs=X_train.shape[1], 
                   num_hidden=hidden_size, 
                   num_outputs=y_train_nn.shape[1])
    # Train the mode using selected hidden size
    train_mse, train_acc, val_mse, val_acc = train(
        model, X_train, y_train_nn, X_val, y_val_nn,
        num_epochs=1000, 
        lr=0.01, 
        batch_size=32)

    # Make predictions on the test set here
    y_test_pred, _ = model.forward(X_test)
    
    test_mse = mse_loss(y_test_nn, y_test_pred)
    test_acc = accuracy(y_test_nn, y_test_pred)
    print(f"\tTest MSE = {test_mse:.3f}\tTest Accuracy = {test_acc:.3f}")

       
    # Add a new row using loc[]
    # Each row represents model with hidden size
    # and it train output
    df.loc[len(df)] = [activation, hidden_size,
                       train_mse, 
                       train_acc, 
                       val_mse, 
                       val_acc,
                       test_mse,
                       test_acc,
                       y_test_pred]




### Question 3 (2 points): For each model, plot the loss/accuracy curves as well as a confusion matrix and ROC-AUC curve for the test dataset.

In [ ]:
# Plot the accuracy and MSE Loss
def plot_acc_mse(train_acc, val_acc, train_mse, val_mse,
                 accuracy_ax, loss_ax):
    
    # Obtain number of epochs
    epochs = range(1, len(train_acc) + 1)

    # To reduce noise just pplot every 0.5% of the epochs
    plot_every = max(1, int(len(train_acc) * 0.005))
    
    # Plot training accuracy
    accuracy_ax.plot(epochs[::plot_every], train_acc[::plot_every], label="Training")
    
    # Plot validation accuracy
    accuracy_ax.plot(epochs[::plot_every], val_acc[::plot_every], label="Validation")
       
    accuracy_ax.set_title(f"Accuracy - {activation} w/Hidden Size {hidden_size}")
    accuracy_ax.set_xlabel("Epoch")
    accuracy_ax.set_ylabel("Accuracy")
    accuracy_ax.grid(True, color='grey', linewidth=1.4, linestyle='-.')
    accuracy_ax.legend()
    # Plot training loss
    loss_ax.plot(epochs[::plot_every], train_mse[::plot_every], label="Training")
    # Plot validation loss
    loss_ax.plot(epochs[::plot_every], val_mse[::plot_every], label="Validation")
    
    loss_ax.set_title(f"MSE Loss - {activation} w/Hidden Size {hidden_size}")
    loss_ax.set_xlabel("Epoch")
    loss_ax.set_ylabel("MSE")
    loss_ax.grid(True, color='grey', linewidth=1.4, linestyle='-.')
    loss_ax.legend()

In [ ]:
# Plot Confusion Matrix and ROC-AUC
def plt_cm_ROC(y_actual, y_predicted, cm_ax, ROC_ax):

    # Both input arrays have shape (N, 1).
    true_labels = y_actual[:, 0]
    prediction_scores = y_predicted[:, 0]

    # Confusion matrix: convert scores to classes.
    predicted_labels = (prediction_scores >= 0.5).astype(int)
    # Create confusion matrix
    cm = confusion_matrix(
        true_labels, predicted_labels, labels=[0, 1]
    )
    # Display confusion matrix
    display = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=[0, 1]
    )

    display.plot(ax=cm_ax, colorbar=False)
    cm_ax.set_title(f"Test Confusion Matrix - {activation} w/Hidden Size {hidden_size}")

    # ROC: use the continuous scores (probabilities)
    fpr, tpr, thresholds = roc_curve(true_labels, prediction_scores)
    # Obtain roc_auc and return it
    roc_auc = auc(fpr, tpr)
    # Display ROC_AUC 
    ROC_ax.plot(fpr, tpr, label=f"ROC curve (AUC = {roc_auc:.3f})")
    ROC_ax.plot(
        [0, 1], [0, 1], "--", color="gray", label="Chance")

    ROC_ax.set_xlim(0.0, 1.0)
    ROC_ax.set_ylim(0.0, 1.05)
    ROC_ax.set_xlabel("False Positive Rate")
    ROC_ax.set_ylabel("True Positive Rate")
    ROC_ax.grid(True, color='grey', linewidth=1.4, linestyle='-.')
    ROC_ax.set_title(f"Test ROC Curve - {activation} w/Hidden Size {hidden_size}")
    ROC_ax.legend()

    return roc_auc

In [ ]:
# Index thru the dataframe that has the stored results
for i in df.index:
    hidden_size = df.loc[i, 'hidden_size']
    activation = df.loc[i, 'activation']
    # Create a 2 x 2 figure
    fig, ax = plt.subplots(2, 2, figsize=(16, 12))

    # Top row: this model's accuracy and loss histories.
    plot_acc_mse(
        train_acc=df.loc[i, 'train_acc'],
        val_acc=df.loc[i, 'val_acc'],
        train_mse=df.loc[i, 'train_mse'],
        val_mse=df.loc[i, 'val_mse'],
        accuracy_ax=ax[0, 0],
        loss_ax=ax[0, 1]
    )

    # Bottom row: this model's test confusion matrix and ROC.
    roc_auc = plt_cm_ROC(
        y_actual=y_test_nn,
        y_predicted=df.loc[i, 'y_test_pred'],
        cm_ax=ax[1, 0],
        ROC_ax=ax[1, 1]
    )
    # Save the roc_auc score for use in a later question
    df.loc[i, 'test_auc'] = roc_auc

    plt.tight_layout()
    # Save figure if needed
    plt.savefig(
        f"Assignment1_P2_Part3_Q3_{activation}_{hidden_size}_{i}.png"
    )
    plt.show()

### Question 4 (2 points): For each model, compute the total number of trainable parameters. 

Plot the test accuracy and test ROC-AUC as a function of the model size. What trends emerge? Why do you think they occur?

### What are primary trainable parameters of a neural network?

**Weights and Biases**

Trainable parameters means the parameters which can be learned and adjusted during the training process. These parameters need to be optimized as they impact network’s accuracy and performance.


$$\text{Total parameters}=4H+H+H+1=6H+1$$

| Hidden neurons | Total trainable parameters |
|---:|---:|
| 16 | 97 |
| 32 | 193 |
| 64 | 385 |
| 128 | 769 |
| 256 | 1,537 |
| 512 | 3,073 |
| 1,024 | 6,145 |

In [ ]:
model_sizes = [
    (16, 97),
    (32, 193),
    (64, 385),
    (128, 769),
    (256, 1537),
    (512, 3073),
    (1024, 6145)
]

marker_positions = list(map(itemgetter(-1), model_sizes))

parameter_counts = []
test_accuracies = []
test_aucs = []

for hidden_size, parameter_count in model_sizes:
    row = df.loc[df["hidden_size"] == hidden_size].iloc[0]
    parameter_counts.append(parameter_count)
    test_accuracies.append(row["test_acc"])
    test_aucs.append(row["test_auc"])

# Use strings to give the bars equally spaced positions.
labels = [str(count) for count in parameter_counts]

plt.figure(figsize=(10, 5))

plt.plot(
    labels, test_accuracies,
    label="Test Accuracy",
    marker='o', linestyle='-')

# add in the values
for i in range(len(test_accuracies)):
    plt.text(i, test_accuracies[i]-0.01, str(round(test_accuracies[i],3)), 
             horizontalalignment = 'center',             
             fontsize = 10)
    
plt.xlabel("Number of Trainable Parameters")
plt.ylabel("Test Accuracy")
plt.grid(True)
plt.title(f"{activation} Test Accuracy by Total Number of Trainable Parameters")
plt.savefig("Assignment1_P2_Part3_Q4a.png")
plt.show()

# Use strings to give the bars equally spaced positions.
labels = [str(count) for count in parameter_counts]

plt.figure(figsize=(10, 5))

plt.plot(
    labels, test_aucs,
    label="Test ROC-AUC",
    marker='o', linestyle='-')

# add in the values
for i in range(len(test_aucs)):
    plt.text(i, test_aucs[i]-0.01, str(round(test_aucs[i],3)), 
             horizontalalignment = 'center',             
             fontsize = 10)
    
plt.xlabel("Number of Trainable Parameters")
plt.ylabel("Test ROC-AUC")
plt.grid(True)
plt.title(f"{activation} Test ROC-AUC by Total Number of Trainable Parameters")
plt.savefig("Assignment1_P2_Part3_Q4b.png")
plt.show()


### What trends emerge? Why do you think they occur?

Plots show a fairly clear trend, and yes, accuracy improves as the model grows initially, but it does not continue improving indefinitely. After a certain point just adding neurons in the hidden layer doesn't improve accuracy.

ROC-AUC follows a similar pattern although it reaches its maximum at a much higher trainable parameter level (769 vs 3073). However, it drops significantly when the number of trainable paramerters is increased. 

As to why, increasing the hidden-layer width gives the network more parameters and therefore greater capacity to learn the nonlinear decision boundary in this dataset. However, once the model has enough capacity to capture patterns, just adding neurons actually results in diminishing returns. Overfitting on the training data also comes to mind.


## Part 4: Activation functions (14 points)
Question 1 (4 points): Modify your neural network by replacing ReLU with a “Leaky ReLU"

### Question 1 (4 points): Modify your neural network by replacing ReLU with a “Leaky ReLU"

$$
\operatorname{Leaky  ReLU}(z) = \left\{\begin{matrix}
z &z\geq 0  \\ 
\lambda z &z< 0  \\
\end{matrix}\right.
$$

$$
\operatorname{LeakyReLU}'(z)
=
\begin{cases}
1, &z\geq 0  \\ 
\lambda, & z < 0.
\end{cases}
$$

Plot this activation function and compute its derivative to modify the backward pass. Continue to use tanh on the output layer. Train the Leaky ReLU model using a hyperparameter configuration from Part 3, and plot training/validation loss/accuracy curves, a test confusion matrix, and a test ROC-AUC curve.

### Leaky ReLU

In [ ]:
# Leaky ReLU 
# leaky_relu used in forward pass
# grad_leaky_relu is the derivative of leaky_relu
# and is used in backward pass

def leaky_relu(z, lambda_=0.01):
    result = np.where(z >= 0, z, lambda_ * z)
    return result

def grad_leaky_relu(z, lambda_=0.01):
    result = np.where(z >= 0, 1, lambda_)
    return result

### Leaky ReLU

In [ ]:
z = np.linspace(-5, 5, 100)

plt.plot(z, leaky_relu(z))
plt.xlabel("z")
plt.ylabel("Leaky ReLU(z)")
plt.title("Leaky ReLU Activation Function")
plt.show()

plt.plot(z, grad_leaky_relu(z))
plt.xlabel("z")
plt.ylabel("d(Leaky ReLU)/dz")
plt.title("Derivative of Leaky ReLU")
plt.show()

### Leaky ReLU

In [ ]:
class MLP_2Layer_LeakyReLU:
    def __init__(self, 
                 num_inputs,
                 num_hidden,
                 num_outputs,
                 random_seed=42):
        self.num_inputs = num_inputs
        self.num_hidden = num_hidden
        self.num_outputs = num_outputs
        self.random_seed = random_seed
        self._initialize_weights()

    def _initialize_weights(self):
        np.random.seed(self.random_seed)

        # Initialize hidden layer weights and biases
        self.w_hidden = np.random.normal(loc=0, scale=0.1,
                                         size=(self.num_inputs, self.num_hidden))
        self.b_hidden = np.zeros(self.num_hidden)

        # Initialize output layer weights and biases
        self.w_output = np.random.normal(loc=0, scale=0.1,
                                         size=(self.num_hidden, self.num_outputs))
        self.b_output = np.zeros(self.num_outputs)

    def forward(self, x):
        """ Feed input data x through each network layer 
            Return a final network output, as well as the intermediate
            values at each layer
        """
        # Apply hidden layer functions
        z_hidden = preactivation(x, self.w_hidden, self.b_hidden)
        
        # Using leaky relu
        a_hidden = leaky_relu(z_hidden)

        # Apply output layer function
        z_output = preactivation(a_hidden, self.w_output, self.b_output)
        
        # Using tanh instead of sigmoid
        a_output = np.tanh(z_output)


        return a_output, (
            a_output,
            z_output,
            a_hidden,
            z_hidden,
        )

    def backward(self, x, y_true, a_output, z_output, a_hidden, z_hidden):
        """ Compute gradients with respect to each model parameter """
        # Sanity check on input shapes
        B, I = x.shape # [B, I]
        _, O = y_true.shape # [B, O]
        assert y_true.shape[0] == B

        # Compute gradients w.r.t. output layer preactivation
        # Should be same size as z_output: [B, O]
        d_loss__d_a_output = grad_mse_loss(y_true, a_output) #[B, O]
        
        d_a_output__d_z_output = 1.0 - a_output**2  # Shape: (B, O)

        d_loss__d_z_output = d_loss__d_a_output * d_a_output__d_z_output #[B, O]

        # Compute gradients w.r.t. output layer weights
        # Should be same size as w_output: [H, O]
        d_z_output__d_w_output = a_hidden #[B, H]
        d_loss__d_w_output = np.einsum('Bo,Bh->ho', d_loss__d_z_output, d_z_output__d_w_output)

        # Compute gradients w.r.t. output layer biases
        # Should be same size as b_output: [O,]
        d_z_output__d_b_output = 1.
        d_loss__d_b_output = np.einsum('Bo,->o', d_loss__d_z_output, d_z_output__d_b_output)

        # Compute gradients w.r.t. hidden layer preactivation
        # Should be same size as z_hidden: [B, H]
        d_z_output__d_a_hidden = self.w_output #[H, O]
        d_loss__d_a_hidden = np.einsum('Bo,ho->Bh', d_loss__d_z_output, d_z_output__d_a_hidden)

        # Using grad_leaky_relu
        # Derivative of leaky_relu
        d_a_hidden__d_z_hidden = grad_leaky_relu(z_hidden) #[B, H]
        
        d_loss__d_z_hidden = d_loss__d_a_hidden * d_a_hidden__d_z_hidden #[B, H]

        # Compute gradients w.r.t. hidden layer weights
        # Should be same size as w_hidden: [I, H]
        d_z_hidden__d_w_hidden = x #[B, I]
        d_loss__d_w_hidden = np.einsum('Bh,Bi->ih', d_loss__d_z_hidden, d_z_hidden__d_w_hidden)

        # Compute gradients w.r.t. hidden layer biases
        # Should be same size as b_hidden: [H,]
        d_z_hidden__d_b_hidden = 1.
        d_loss__d_b_hidden = np.einsum('Bh,->h', d_loss__d_z_hidden, d_z_hidden__d_b_hidden)

        # We're done. Return all the gradients w.r.t. weights and biases
        return d_loss__d_w_output, \
               d_loss__d_b_output, \
               d_loss__d_w_hidden, \
               d_loss__d_b_hidden

    def weight_update(self, d_loss__d_w_output, d_loss__d_b_output, d_loss__d_w_hidden, d_loss__d_b_hidden, lr=0.3):
        """ Update weights and biases via gradient descent
        """
        self.w_output -= lr * d_loss__d_w_output
        self.b_output -= lr * d_loss__d_b_output
        self.w_hidden -= lr * d_loss__d_w_hidden
        self.b_hidden -= lr * d_loss__d_b_hidden

### Leaky ReLU

In [ ]:

for hidden_size in [512]:
    activation = "Leaky ReLU"
    print(f"{activation} MLP w/Hidden Size {hidden_size}")
    model = MLP_2Layer_LeakyReLU(num_inputs=X_train.shape[1], 
                   num_hidden=hidden_size, 
                   num_outputs=y_train_nn.shape[1])

    train_mse, train_acc, val_mse, val_acc = train(
        model, X_train, y_train_nn, X_val, y_val_nn,
        num_epochs=1000, 
        lr=0.01, 
        batch_size=32)

    # Make predictions on the test set here
    y_test_pred, _ = model.forward(X_test)
    
    test_mse = mse_loss(y_test_nn, y_test_pred)
    test_acc = accuracy(y_test_nn, y_test_pred)
    print(f"\tTest MSE = {test_mse:.3f}\tTest Accuracy = {test_acc:.3f}")

       
    # Add a new row using loc[]
    # Each row represents model with hidden size
    # and it train output
    df.loc[len(df)] = ['Leaky ReLU', hidden_size,
                       train_mse, 
                       train_acc, 
                       val_mse, 
                       val_acc,
                       test_mse,
                       test_acc,
                       y_test_pred,0
                      ]




### Leaky ReLU

In [ ]:
for i in df.index:
    if df.loc[i, "activation"] == "Leaky ReLU":
        hidden_size = df.loc[i, 'hidden_size']
    
        fig, ax = plt.subplots(2, 2, figsize=(20, 12))
        # ax[0] → Training and validation accuracy 
        # ax[1] → Training and loss
        plot_acc_mse(
            train_acc, val_acc, train_mse, val_mse,
            accuracy_ax=ax[0,0],
            loss_ax=ax[0,1])
    
        # ax[2] → Test confusion matrix
        # ax[3] → Test ROC curve
            
        # Read this model's saved test predictions.
        roc_auc = plt_cm_ROC(
            y_actual=y_test_nn,
            y_predicted=df.loc[i, 'y_test_pred'],
            cm_ax=ax[1,0],
            ROC_ax=ax[1,1]
        )
    
        # Save AUC
        df.loc[i, 'test_auc'] = roc_auc
        
        plt.tight_layout()
    
        plt.savefig("Assignment1_P2_Part4_Q1.png")
        plt.show()
        

### Question 2 (4 points): Modify your neural network by replacing ReLU with a “Quadratic"

$$ f(z) = z^2$$

Plot this activation function and compute its derivative to modify the backward pass. Continue to use tanh on the output layer. Train the Quadratic model using a hyperparameter configuration from Part 3. You may need to adjust learning rates or weight initialization to improve performance.

Plot training/validation loss/accuracy curves, a test confusion matrix, and a test ROC-AUC curve.

### Quadratic

$$ f(z) = z^2$$

Using the power rule: 
$$ f'(z) = 2z$$

In [ ]:
# Quadratic
# quadratic(z) used in forward pass
# grad_quadratic(z) is the derivative of quadratic 
# and is used in backward pass

def quadratic(z):
    result = z ** 2
    return result

def grad_quadratic(z):
    result = 2 * z
    return result

### Quadratic

In [ ]:
z = np.linspace(-5, 5, 100)

plt.plot(z, quadratic(z))
plt.xlabel("z")
plt.ylabel("Quadratic(z)")
plt.title("Quadratic Activation Function")
plt.show()

plt.plot(z, grad_quadratic(z))
plt.xlabel("z")
plt.ylabel("d(Quadratic)/dz")
plt.title("Derivative of Quadratic")

### Quadratic

In [ ]:
class MLP_2Layer_Quadratic:
    def __init__(self, 
                 num_inputs,
                 num_hidden,
                 num_outputs,
                 random_seed=42):
        self.num_inputs = num_inputs
        self.num_hidden = num_hidden
        self.num_outputs = num_outputs
        self.random_seed = random_seed
        self._initialize_weights()

    def _initialize_weights(self):
        np.random.seed(self.random_seed)

        # Initialize hidden layer weights and biases
        self.w_hidden = np.random.normal(loc=0, scale=0.1,
                                         size=(self.num_inputs, self.num_hidden))
        self.b_hidden = np.zeros(self.num_hidden)

        # Initialize output layer weights and biases
        self.w_output = np.random.normal(loc=0, scale=0.1,
                                         size=(self.num_hidden, self.num_outputs))
        self.b_output = np.zeros(self.num_outputs)

    def forward(self, x):
        """ Feed input data x through each network layer 
            Return a final network output, as well as the intermediate
            values at each layer
        """
        # Apply hidden layer functions
        z_hidden = preactivation(x, self.w_hidden, self.b_hidden)
        
        # Change to use quadratic
        a_hidden = quadratic(z_hidden)

        # Apply output layer function
        z_output = preactivation(a_hidden, self.w_output, self.b_output)
        
        # Using tanh instead of sigmoid
        a_output = np.tanh(z_output)


        return a_output, (
            a_output,
            z_output,
            a_hidden,
            z_hidden,
        )

    def backward(self, x, y_true, a_output, z_output, a_hidden, z_hidden):
        """ Compute gradients with respect to each model parameter """
        # Sanity check on input shapes
        B, I = x.shape # [B, I]
        _, O = y_true.shape # [B, O]
        assert y_true.shape[0] == B

        # Compute gradients w.r.t. output layer preactivation
        # Should be same size as z_output: [B, O]
        d_loss__d_a_output = grad_mse_loss(y_true, a_output) #[B, O]
        
        d_a_output__d_z_output = 1.0 - a_output**2  # Shape: (B, O)

        d_loss__d_z_output = d_loss__d_a_output * d_a_output__d_z_output #[B, O]

        # Compute gradients w.r.t. output layer weights
        # Should be same size as w_output: [H, O]
        d_z_output__d_w_output = a_hidden #[B, H]
        d_loss__d_w_output = np.einsum('Bo,Bh->ho', d_loss__d_z_output, d_z_output__d_w_output)

        # Compute gradients w.r.t. output layer biases
        # Should be same size as b_output: [O,]
        d_z_output__d_b_output = 1.
        d_loss__d_b_output = np.einsum('Bo,->o', d_loss__d_z_output, d_z_output__d_b_output)

        # Compute gradients w.r.t. hidden layer preactivation
        # Should be same size as z_hidden: [B, H]
        d_z_output__d_a_hidden = self.w_output #[H, O]
        d_loss__d_a_hidden = np.einsum('Bo,ho->Bh', d_loss__d_z_output, d_z_output__d_a_hidden)

        # Using grad_quadratic
        d_a_hidden__d_z_hidden = grad_quadratic(z_hidden) #[B, H]
        
        d_loss__d_z_hidden = d_loss__d_a_hidden * d_a_hidden__d_z_hidden #[B, H]

        # Compute gradients w.r.t. hidden layer weights
        # Should be same size as w_hidden: [I, H]
        d_z_hidden__d_w_hidden = x #[B, I]
        d_loss__d_w_hidden = np.einsum('Bh,Bi->ih', d_loss__d_z_hidden, d_z_hidden__d_w_hidden)

        # Compute gradients w.r.t. hidden layer biases
        # Should be same size as b_hidden: [H,]
        d_z_hidden__d_b_hidden = 1.
        d_loss__d_b_hidden = np.einsum('Bh,->h', d_loss__d_z_hidden, d_z_hidden__d_b_hidden)

        # We're done. Return all the gradients w.r.t. weights and biases
        return d_loss__d_w_output, \
               d_loss__d_b_output, \
               d_loss__d_w_hidden, \
               d_loss__d_b_hidden

    def weight_update(self, d_loss__d_w_output, d_loss__d_b_output, d_loss__d_w_hidden, d_loss__d_b_hidden, lr=0.3):
        """ Update weights and biases via gradient descent
        """
        self.w_output -= lr * d_loss__d_w_output
        self.b_output -= lr * d_loss__d_b_output
        self.w_hidden -= lr * d_loss__d_w_hidden
        self.b_hidden -= lr * d_loss__d_b_hidden

### Quadratic

In [ ]:

for hidden_size in [512]:
    activation = "Quadratic"
    print(f"{activation} MLP w/Hidden Size {hidden_size}")
    model = MLP_2Layer_Quadratic(num_inputs=X_train.shape[1], 
                   num_hidden=hidden_size, 
                   num_outputs=y_train_nn.shape[1])

    train_mse, train_acc, val_mse, val_acc = train(
        model, X_train, y_train_nn, X_val, y_val_nn,
        num_epochs=1000, 
        lr=0.01, 
        batch_size=32)

    # Make predictions on the test set here
    y_test_pred, _ = model.forward(X_test)
    
    test_mse = mse_loss(y_test_nn, y_test_pred)
    test_acc = accuracy(y_test_nn, y_test_pred)
    print(f"\tTest MSE = {test_mse:.3f}\tTest Accuracy = {test_acc:.3f}")

       
    # Add a new row using loc[]
    # Each row represents model with hidden size
    # and it train output
    df.loc[len(df)] = [activation, hidden_size,
                       train_mse, 
                       train_acc, 
                       val_mse, 
                       val_acc,
                       test_mse,
                       test_acc,
                       y_test_pred,0
                      ]



### Quadratic

In [ ]:
for i in df.index:
    if df.loc[i, "activation"] == "Quadratic":
        hidden_size = df.loc[i, 'hidden_size']
    
        fig, ax = plt.subplots(2, 2, figsize=(20, 12))
        # ax[0,0] → Training and validation accuracy 
        # ax[0,1] → Training and loss
        plot_acc_mse(
            train_acc, val_acc, train_mse, val_mse,
            accuracy_ax=ax[0,0],
            loss_ax=ax[0,1])
    
        # ax[1,0] → Test confusion matrix
        # ax[1,1] → Test ROC curve
            
        # Read this model's saved test predictions.
        roc_auc = plt_cm_ROC(
            y_actual=y_test_nn,
            y_predicted=df.loc[i, 'y_test_pred'],
            cm_ax=ax[1,0],
            ROC_ax=ax[1,1]
        )
    
        # Save AUC
        df.loc[i, 'test_auc'] = roc_auc
        
        plt.tight_layout()
    
        plt.savefig("Assignment1_P2_Part4_Q2.png")
        plt.show()
        

### Question 3 (4 points): Modify your neural network by replacing ReLU with a “Swish"

$$
\operatorname{Swish}(z) = \frac{z}{1 + e^{-z}} 
$$

Plot this activation function and compute its derivative to modify the backward pass. Continue to use tanh on the output layer. Train the Swish model using a hyperparameter configuration from Part 3. You may need to adjust learning rates or weight initialization to improve performance. 

Plot training/validation loss/accuracy curves, a test confusion matrix, and a test ROC-AUC curve.

### "Swish"

$$
\operatorname{Swish}(z) = \frac{z}{1 + e^{-z}} 
$$

Calculate the derivative of "Swish":
$$u(z) = z$$
$$v(z) = 1 + e^{-z}$$

$$u'(z) = 1$$
$$v'(z) = -e^{-z}$$

$$f'(z) = \frac{u'v-uv'}{(v)^{2}}$$

$$f'(z) = \frac{ (1)(1 + e^{-z} - (z)(-e^{-z}) }{ (1 + e^{-z})^{2} }$$

$$
\operatorname{Swish}'(z)
=
\frac{1+e^{-z}+ze^{-z}}{(1+e^{-z})^2}
$$

In [ ]:
# Swish 
# swish(z) used in forward pass
# grad_swish(z) is the derivative of swish
# and is used in backward pass

def swish(z):
    result = z / (1 + np.exp(-z))
    return result

def grad_swish(z):
    # Derivative of swish
    # Painful to calculate
    result = (np.exp(z) * (1 + np.exp(z) + z)) / (1 + np.exp(z))**2
    return result



In [ ]:
z = np.linspace(-5, 5, 100)

plt.plot(z, swish(z))
plt.xlabel("z")
plt.ylabel("Swish(z)")
plt.title("Swish Activation Function")
plt.show()

plt.plot(z, grad_swish(z))
plt.xlabel("z")
plt.ylabel("d(Swish)/dz")
plt.title("Derivative of Swish")

### "Swish"

In [ ]:
class MLP_2Layer_Swish:
    def __init__(self, 
                 num_inputs,
                 num_hidden,
                 num_outputs,
                 random_seed=42):
        self.num_inputs = num_inputs
        self.num_hidden = num_hidden
        self.num_outputs = num_outputs
        self.random_seed = random_seed
        self._initialize_weights()

    def _initialize_weights(self):
        np.random.seed(self.random_seed)

        # Initialize hidden layer weights and biases
        self.w_hidden = np.random.normal(loc=0, scale=0.1,
                                         size=(self.num_inputs, self.num_hidden))
        self.b_hidden = np.zeros(self.num_hidden)

        # Initialize output layer weights and biases
        self.w_output = np.random.normal(loc=0, scale=0.1,
                                         size=(self.num_hidden, self.num_outputs))
        self.b_output = np.zeros(self.num_outputs)

    def forward(self, x):
        """ Feed input data x through each network layer 
            Return a final network output, as well as the intermediate
            values at each layer
        """
        # Apply hidden layer functions
        z_hidden = preactivation(x, self.w_hidden, self.b_hidden)
        
        # Using swish formula
        a_hidden = swish(z_hidden)

        # Apply output layer function
        z_output = preactivation(a_hidden, self.w_output, self.b_output)
        
        # Using tanh instead of sigmoid
        a_output = np.tanh(z_output)


        return a_output, (
            a_output,
            z_output,
            a_hidden,
            z_hidden,
        )

    def backward(self, x, y_true, a_output, z_output, a_hidden, z_hidden):
        """ Compute gradients with respect to each model parameter """
        # Sanity check on input shapes
        B, I = x.shape # [B, I]
        _, O = y_true.shape # [B, O]
        assert y_true.shape[0] == B

        # Compute gradients w.r.t. output layer preactivation
        # Should be same size as z_output: [B, O]
        d_loss__d_a_output = grad_mse_loss(y_true, a_output) #[B, O]
        
        d_a_output__d_z_output = 1.0 - a_output**2  # Shape: (B, O)

        d_loss__d_z_output = d_loss__d_a_output * d_a_output__d_z_output #[B, O]

        # Compute gradients w.r.t. output layer weights
        # Should be same size as w_output: [H, O]
        d_z_output__d_w_output = a_hidden #[B, H]
        d_loss__d_w_output = np.einsum('Bo,Bh->ho', d_loss__d_z_output, d_z_output__d_w_output)

        # Compute gradients w.r.t. output layer biases
        # Should be same size as b_output: [O,]
        d_z_output__d_b_output = 1.
        d_loss__d_b_output = np.einsum('Bo,->o', d_loss__d_z_output, d_z_output__d_b_output)

        # Compute gradients w.r.t. hidden layer preactivation
        # Should be same size as z_hidden: [B, H]
        d_z_output__d_a_hidden = self.w_output #[H, O]
        d_loss__d_a_hidden = np.einsum('Bo,ho->Bh', d_loss__d_z_output, d_z_output__d_a_hidden)

        # Using grad_swish
        # Derivative of swish equation
        d_a_hidden__d_z_hidden = grad_swish(z_hidden) #[B, H]
        
        d_loss__d_z_hidden = d_loss__d_a_hidden * d_a_hidden__d_z_hidden #[B, H]

        # Compute gradients w.r.t. hidden layer weights
        # Should be same size as w_hidden: [I, H]
        d_z_hidden__d_w_hidden = x #[B, I]
        d_loss__d_w_hidden = np.einsum('Bh,Bi->ih', d_loss__d_z_hidden, d_z_hidden__d_w_hidden)

        # Compute gradients w.r.t. hidden layer biases
        # Should be same size as b_hidden: [H,]
        d_z_hidden__d_b_hidden = 1.
        d_loss__d_b_hidden = np.einsum('Bh,->h', d_loss__d_z_hidden, d_z_hidden__d_b_hidden)

        # We're done. Return all the gradients w.r.t. weights and biases
        return d_loss__d_w_output, \
               d_loss__d_b_output, \
               d_loss__d_w_hidden, \
               d_loss__d_b_hidden

    def weight_update(self, d_loss__d_w_output, d_loss__d_b_output, d_loss__d_w_hidden, d_loss__d_b_hidden, lr=0.3):
        """ Update weights and biases via gradient descent
        """
        self.w_output -= lr * d_loss__d_w_output
        self.b_output -= lr * d_loss__d_b_output
        self.w_hidden -= lr * d_loss__d_w_hidden
        self.b_hidden -= lr * d_loss__d_b_hidden

### "Swish"

In [ ]:
for hidden_size in [512]:
    activation = "Swish"
    print(f"{activation} MLP w/Hidden Size {hidden_size}")
    model = MLP_2Layer_Swish(num_inputs=X_train.shape[1], 
                   num_hidden=hidden_size, 
                   num_outputs=y_train_nn.shape[1])

    train_mse, train_acc, val_mse, val_acc = train(
        model, X_train, y_train_nn, X_val, y_val_nn,
        num_epochs=1000, 
        lr=0.01, 
        batch_size=32)

    # Make predictions on the test set here
    y_test_pred, _ = model.forward(X_test)
    
    test_mse = mse_loss(y_test_nn, y_test_pred)
    test_acc = accuracy(y_test_nn, y_test_pred)
    print(f"\tTest MSE = {test_mse:.3f}\tTest Accuracy = {test_acc:.3f}")

       
    # Add a new row using loc[]
    # Each row represents model with hidden size
    # and it train output
    df.loc[len(df)] = [activation, hidden_size,
                       train_mse, 
                       train_acc, 
                       val_mse, 
                       val_acc,
                       test_mse,
                       test_acc,
                       y_test_pred,0
                      ]



### "Swish"

In [ ]:
for i in df.index:
    if df.loc[i, "activation"] == "Swish":
        hidden_size = df.loc[i, 'hidden_size']
    
        fig, ax = plt.subplots(2, 2, figsize=(20, 12))
        # ax[0,0] → Training and validation accuracy 
        # ax[0,1] → Training and loss
        plot_acc_mse(
            train_acc, val_acc, train_mse, val_mse,
            accuracy_ax=ax[0,0],
            loss_ax=ax[0,1])
    
        # ax[1,0] → Test confusion matrix
        # ax[1,1] → Test ROC curve
            
        # Read this model's saved test predictions.
        roc_auc = plt_cm_ROC(
            y_actual=y_test_nn,
            y_predicted=df.loc[i, 'y_test_pred'],
            cm_ax=ax[1,0],
            ROC_ax=ax[1,1]
        )
    
        # Save AUC
        df.loc[i, 'test_auc'] = roc_auc
        
        plt.tight_layout()
    
        plt.savefig("Assignment1_P2_Part4_Q3.png")
        plt.show()
        

### Question 4 (2 points): Compare performance of models using different activation functions. 

Did any perform better than the others? Why do you think this might be the case?

In [ ]:
# Display results of model performance
comparison_df = df[
    ["activation", "hidden_size", "test_acc", "test_auc"]
].sort_values(["hidden_size", "activation"])

print("Comparison of Model Activations")
print(comparison_df.to_string(
    index=False,
    formatters={
        "test_acc": "{:.3f}".format,
        "test_auc": "{:.3f}".format,
    }
))

This analysis looks at the results from the four runs with a hidden size of 512. All four used the same learning rate (0.01), batch size (32), and number of epochs (1000).

Swish improved test accuracy over ReLU by 3.6 percentage points, while Quadratic performed substantially worse. Leaky ReLU also performed worse than ReLU. The ReLU results show that increasing hidden size does not consistently improve performance: the 1024-unit model scored below the 512-unit model.

The plots of each activation function and its derivative offer some clues about these behaviors. Swish is smooth and lets small negative inputs contribute, whereas ReLU sets all negative inputs to zero. Quadratic maps inputs of equal magnitude but opposite sign to the same output, so it loses any distinction between them. Its training curves were also very noisy, with large oscillations. This **suggests** that Quadratic may have had difficulty converging with the chosen learning rate and initialization.

These results come from one particular dataset, so they don't mean Swish will always be the best or Quadratic the worst. Models should be able to run with different activation functions, so the best one can be chosen based on validation performance.


## Part 5: Regularization (12 points)

In [ ]:
# Create a MLP class with one hidden layer
# Uses ReLU activation
# and supports L2 and L1 regularization
class MLP_2Layer_Regularized:
    def __init__(self, 
                 num_inputs,
                 num_hidden,
                 num_outputs,
                 l1_weight=0.,
                 l2_weight=0.,
                 random_seed=42):
        
        self.num_inputs = num_inputs
        self.num_hidden = num_hidden
        self.num_outputs = num_outputs
        self.l1_weight = l1_weight
        self.l2_weight = l2_weight
        
        self.random_seed = random_seed
        
        self._initialize_weights()

    def _initialize_weights(self):
        np.random.seed(self.random_seed)

        # Initialize hidden layer weights and biases
        self.w_hidden = np.random.normal(loc=0, scale=0.1,
                                         size=(self.num_inputs, self.num_hidden))
        self.b_hidden = np.zeros(self.num_hidden)

        # Initialize output layer weights and biases
        self.w_output = np.random.normal(loc=0, scale=0.1,
                                         size=(self.num_hidden, self.num_outputs))
        self.b_output = np.zeros(self.num_outputs)

    def forward(self, x):
        """ Feed input data x through each network layer 
            Return a final network output, as well as the intermediate
            values at each layer
        """
        # Apply hidden layer functions
        z_hidden = preactivation(x, self.w_hidden, self.b_hidden)
        # Using relu
        a_hidden = relu(z_hidden)

        # Apply output layer function
        z_output = preactivation(a_hidden, self.w_output, self.b_output)
        # Using tanh instead of sigmoid
        a_output = np.tanh(z_output)


        return a_output, (
            a_output,
            z_output,
            a_hidden,
            z_hidden,
        )

    def backward(self, x, y_true, a_output, z_output, a_hidden, z_hidden):
        """ Compute gradients with respect to each model parameter """
        # Sanity check on input shapes
        B, I = x.shape # [B, I]
        _, O = y_true.shape # [B, O]
        assert y_true.shape[0] == B

        # Compute gradients w.r.t. output layer preactivation
        # Should be same size as z_output: [B, O]
        d_loss__d_a_output = grad_mse_loss(y_true, a_output) #[B, O]
        
        d_a_output__d_z_output = 1.0 - a_output**2  # Shape: (B, O)

        d_loss__d_z_output = d_loss__d_a_output * d_a_output__d_z_output #[B, O]

        # Compute gradients w.r.t. output layer weights
        # Should be same size as w_output: [H, O]
        d_z_output__d_w_output = a_hidden #[B, H]
        d_loss__d_w_output = np.einsum('Bo,Bh->ho', d_loss__d_z_output, d_z_output__d_w_output)

        # Compute gradients w.r.t. output layer biases
        # Should be same size as b_output: [O,]
        d_z_output__d_b_output = 1.
        d_loss__d_b_output = np.einsum('Bo,->o', d_loss__d_z_output, d_z_output__d_b_output)

        # Compute gradients w.r.t. hidden layer preactivation
        # Should be same size as z_hidden: [B, H]
        d_z_output__d_a_hidden = self.w_output #[H, O]
        d_loss__d_a_hidden = np.einsum('Bo,ho->Bh', d_loss__d_z_output, d_z_output__d_a_hidden)

        # Using grad_relu
        # Derivative of relu
        d_a_hidden__d_z_hidden = grad_relu(z_hidden, a_hidden) #[B, H]
        d_loss__d_z_hidden = d_loss__d_a_hidden * d_a_hidden__d_z_hidden #[B, H]

        # Compute gradients w.r.t. hidden layer weights
        # Should be same size as w_hidden: [I, H]
        d_z_hidden__d_w_hidden = x #[B, I]
        d_loss__d_w_hidden = np.einsum('Bh,Bi->ih', d_loss__d_z_hidden, d_z_hidden__d_w_hidden)

        # Compute gradients w.r.t. hidden layer biases
        # Should be same size as b_hidden: [H,]
        d_z_hidden__d_b_hidden = 1.
        d_loss__d_b_hidden = np.einsum('Bh,->h', d_loss__d_z_hidden, d_z_hidden__d_b_hidden)

        
        # L2 regularization
        d_loss__d_w_output += 2 * self.l2_weight * self.w_output
        d_loss__d_w_hidden += 2 * self.l2_weight * self.w_hidden
        
        # L1 regularization
        d_loss__d_w_output += self.l1_weight * np.sign(self.w_output)
        d_loss__d_w_hidden += self.l1_weight * np.sign(self.w_hidden)
        
        
        # We're done. Return all the gradients w.r.t. weights and biases
        return d_loss__d_w_output, \
               d_loss__d_b_output, \
               d_loss__d_w_hidden, \
               d_loss__d_b_hidden

    def weight_update(self, d_loss__d_w_output, d_loss__d_b_output, d_loss__d_w_hidden, d_loss__d_b_hidden, lr=0.3):
        """ Update weights and biases via gradient descent
        """
        self.w_output -= lr * d_loss__d_w_output
        self.b_output -= lr * d_loss__d_b_output
        self.w_hidden -= lr * d_loss__d_w_hidden
        self.b_hidden -= lr * d_loss__d_b_hidden

In [ ]:
# Per values from Part 5 , Question 3
regularization_weights = [0,0.000001,0.00001,0.0001,0.001,0.01,0.1]

In [ ]:
# Create dataframe to store results
df_regularization = pd.DataFrame(columns=[
    # "L1" or "L2"
    "regularization",
    # lambda
    "weight",          
    "train_acc",
    "val_acc",
    "test_acc"
])

### Question 1 (3 points): Using a MLP with at least 256 neurons from Part 3, apply L2 regularization.
Modify the backward pass as needed.

In [ ]:
# Loop thru weights
for weight in regularization_weights:
    regularization = "L2"
    print(f"MLP {regularization} weight {weight}")
    model = MLP_2Layer_Regularized(
            num_inputs=X_train.shape[1], 
            num_hidden=512, 
            num_outputs=y_train_nn.shape[1],
            l1_weight=0,
            l2_weight=weight)

    # train model
    train_mse, train_acc, val_mse, val_acc = train(
        model, X_train, y_train_nn, X_val, y_val_nn,
        num_epochs=1000, 
        lr=0.01, 
        batch_size=32)

    # save train accuracy
    # save validation accuracy
    # save test accuracy

     # Make predictions on the test set here
    y_test_pred, _ = model.forward(X_test)
    
    test_mse = mse_loss(y_test_nn, y_test_pred)
    test_acc = accuracy(y_test_nn, y_test_pred)
    print(f"\tTest MSE = {test_mse:.3f}\tTest Accuracy = {test_acc:.3f}")

       
    # Add a new row using loc[]
    # Each row represents model with L2 and weight
    # save train accuracy
    # save validation accuracy
    # save test accuracy
    df_regularization.loc[len(df_regularization)] = [regularization, 
                       weight,
                       train_acc[-1], 
                       val_acc[-1],
                       test_acc
                      ]



### Question 2 (3 points): Using a MLP with at least 256 neurons from Part 3, apply L1 regularization. 

Modify the backward pass as needed.


In [ ]:
# Loop thru weights
for weight in regularization_weights:
    regularization = "L1"
    print(f"MLP {regularization} weight {weight}")
    model = MLP_2Layer_Regularized(
            num_inputs=X_train.shape[1], 
            num_hidden=512, 
            num_outputs=y_train_nn.shape[1],
            l1_weight=weight,
            l2_weight=0)

    # train model
    train_mse, train_acc, val_mse, val_acc = train(
        model, X_train, y_train_nn, X_val, y_val_nn,
        num_epochs=1000, 
        lr=0.01, 
        batch_size=32)


    # Make predictions on the test set here
    y_test_pred, _ = model.forward(X_test)
    
    test_mse = mse_loss(y_test_nn, y_test_pred)
    test_acc = accuracy(y_test_nn, y_test_pred)
    print(f"\tTest MSE = {test_mse:.3f}\tTest Accuracy = {test_acc:.3f}")

       
    # Add a new row using loc[]
    # Each row represents model with L1 and weight
    # save train accuracy
    # save validation accuracy
    # save test accuracy
    df_regularization.loc[len(df_regularization)] = [regularization, 
                       weight,
                       train_acc[-1], 
                       val_acc[-1],
                       test_acc
                      ]



### Question 3 (3 points): For both L1 and L2 regularization, train models with weights

$$\lambda \in  [0, 10^{−6}, 10^{−5}, 10^{−4}, 10^{−3}, 10^{−2}, 10^{−1}]$$

Plot the training, validation, and test accuracy as a function of regularization strength.



In [ ]:
df_regularization

In [ ]:
def plot_regularization_accuracy(df, regularization, ax):
    """
    Plot train, validation, and test accuracy against regularization strength.

    Filters the results to a single regularization type (e.g., "L1" or "L2"),
    then draws one line per data split on the provided axes.

    Parameters
    ----------
    df : pandas.DataFrame
        Experiment results. Must contain the columns "regularization",
        "weight", "train_acc", "val_acc", and "test_acc".
    regularization : str
        The regularization type to plot (matched against df["regularization"]).
    ax : matplotlib.axes.Axes
        The axes to draw on. The caller creates it, which makes it easy to
        place several plots side by side in one figure.

    Returns
    -------
    None
        The plot is drawn directly onto `ax`.
    """
    # Keep only the rows for the requested regularization type, and sort by
    # weight so the lines connect points from left to right.
    results = df[
        df["regularization"] == regularization
    ].sort_values("weight")

    # Plot accuracy for each data split against regularization strength.
    # Markers show the individual weight values that were actually tested.
    ax.plot(results["weight"], results["train_acc"],
            marker="o", label="Training")
    ax.plot(results["weight"], results["val_acc"],
            marker="o", label="Validation")
    ax.plot(results["weight"], results["test_acc"],
            marker="o", label="Test")

    # Supports lambda=0 and spreads out the small positive values.
    # (A plain log scale can't display 0; symlog is linear within
    # +/- linthresh and logarithmic beyond it.)
    ax.set_xscale("symlog", linthresh=1e-6)

    # Place a tick at every tested weight, and label it explicitly:
    # "0" for the unregularized case, scientific notation (e.g., 1e-03)
    # for everything else. Labels are rotated to avoid overlapping.
    ax.set_xticks(results["weight"])
    ax.set_xticklabels(
        ["0" if w == 0 else f"{w:.0e}" for w in results["weight"]],
        rotation=45
    )

    # Axis labels and title identify what is being shown.
    ax.set_xlabel("Regularization strength (λ)")
    ax.set_ylabel("Accuracy")
    ax.set_title(f"{regularization} Regularization")

    # Legend distinguishes the three lines; a light grid aids reading values
    # without dominating the plot.
    ax.legend()
    ax.grid(True, alpha=0.3)

### Comparison of L1 and L2 Regularization Results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

plot_regularization_accuracy(df_regularization, "L1", axes[0])
plot_regularization_accuracy(df_regularization, "L2", axes[1])

plt.tight_layout()
plt.show()

In [ ]:
# List regularization values
print(df_regularization.sort_values(
    by=["regularization", "weight"]
).to_string(index=False))

### Question 4 (3 points): Comment on how regularization affects model performance. Why do you think regularization impacts overfitting in each case?


### Overview of Regularization Techniques
- Ridge (L2): Shrinks coefficients toward zero but does not make them exactly zero, so all features stay in the model.
- Lasso (L1): Performs automatic feature selection by driving some coefficients to zero. This discards the less important features and keeps the ones that matter most.

Regularization improved performance at small strengths, but stronger regularization reduced accuracy substantially.

For L1, λ = 0.0001 produced the best results. Validation accuracy increased from 90.98% without regularization to 93.66%, and test accuracy increased from 91.95% to 95.12%.
The gap between training and validation accuracy also decreased, from about 4.30 to 3.33 percentage points. **This suggests that L1 reduced overfitting at this strength**, even though training accuracy also improved.

For L2, λ = 0.00001 produced the highest validation accuracy (91.71%) and test accuracy (93.41%). These improvements were smaller than those from L1.
The training–validation gap decreased only slightly, from about 4.30 to 4.06 percentage points, **suggesting a modest reduction in overfitting**.

Both methods discourage large weights, but at λ = 0.01 and 0.1, training, validation, and test accuracies were all near chance accuracy. This indicates underfitting: the penalty was too strong for the model to learn the patterns effectively.

Overall, these runs suggest that appropriately chosen regularization can improve generalization, but more regularization does not always produce better performance.
Both methods should be tried to see which performs best on the validation data. Sometimes L1 will come out ahead and other times L2 will. It depends heavily on the data being used.
